# FX Sentiment — Model Training Notebook v3
# Run Cell 0, restart runtime, then run all remaining cells top to bottom.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 0 — Install (run ONCE, then RESTART runtime)  ║
# ╚══════════════════════════════════════════════════════╝
# After restart: Runtime → Restart Runtime → then run Cell 1 onwards.
import subprocess, sys

packages = [
    "torch",
    "transformers>=4.30,<5.0",   # 4.30–4.46 — works on any Python version
    "datasets>=2.0",
    "scikit-learn>=1.2,<1.4",    # 1.3.x is stable on Python 3.8/3.9/3.10/3.11
    "lightgbm>=4.0",
    "pandas>=1.5",
    "numpy>=1.23,<2.0",          # <2.0 ensures protocol=2 pickle cross-compat
    "matplotlib>=3.6",
    "seaborn>=0.12",
    "joblib>=1.2",
    "scipy>=1.9",
    "feedparser>=6.0",           # for RSS news (used in predict.py)
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
print("\n✓ Installation complete.  NOW: Runtime → Restart Runtime")
print("  After restart, run Cell 1 onwards — do NOT re-run this cell.")


## Cell 1 — Imports & constants

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports & constants                       ║
# ╚══════════════════════════════════════════════════════╝
from __future__ import annotations   # Python 3.8 compat for type hints
import warnings, joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

import lightgbm as lgb
from scipy.sparse import hstack, csr_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
sns.set_style("darkgrid")
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11})

DATA_PATH  = Path("data/output/combined_zenodo_fx_sentiment.csv")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

LABEL2ID   = {"buy": 0, "hold": 1, "sell": 2}
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}
CURRENCIES = ["USD", "EUR", "JPY", "CHF", "GBP"]
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

def safe_dump(obj, path):
    """Save with protocol=2 — loads on numpy 1.x AND 2.x without any patching."""
    joblib.dump(obj, path, protocol=2, compress=3)
    size_mb = Path(path).stat().st_size / 1e6
    print(f"  Saved → {path}  ({size_mb:.1f} MB)")

print(f"NumPy      : {np.__version__}")
print(f"Device     : {DEVICE}")
print(f"Models dir : {MODELS_DIR.resolve()}")
print("\n✓ Imports done.")


## Cell 2 — Load & clean dataset

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 2 — Load & clean dataset                      ║
# ╚══════════════════════════════════════════════════════╝
assert DATA_PATH.exists(), f"File not found: {DATA_PATH}\nMake sure combined_zenodo_fx_sentiment.csv is in data/output/"

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Raw shape  : {df.shape}")
print(f"Columns    : {list(df.columns)[:10]} ...")

# ── build raw_text (title + text if available) ────────────────────────────────
if "text" in df.columns and "title" in df.columns:
    df["raw_text"] = (df["title"].fillna("").astype(str) + " " +
                      df["text"].fillna("").astype(str)).str.strip()
elif "text" in df.columns:
    df["raw_text"] = df["text"].fillna("").astype(str)
elif "title" in df.columns:
    df["raw_text"] = df["title"].fillna("").astype(str)
else:
    str_cols = df.select_dtypes("object").columns.tolist()
    assert str_cols, "No text column found in the dataset."
    df["raw_text"] = df[str_cols[0]].fillna("").astype(str)

# ── target label ──────────────────────────────────────────────────────────────
# Use 'impact' column as label (buy / sell / hold)
df["impact"] = df["impact"].astype(str).str.lower().str.strip()
df = df[df["impact"].isin(["buy", "sell", "hold"])].copy()
df = df[df["currency"].isin(CURRENCIES)].copy()
df = df[df["raw_text"].str.len() > 15].copy()

# ── finbert score (optional feature) ──────────────────────────────────────────
if "finbert_sent_score" in df.columns:
    df["finbert_score"] = pd.to_numeric(df["finbert_sent_score"], errors="coerce").fillna(0.0)
else:
    df["finbert_score"] = 0.0

df = df.reset_index(drop=True)
df["label_id"] = df["impact"].map(LABEL2ID)

print(f"\nClean shape : {df.shape}")
print(f"\nLabel distribution:\n{df['impact'].value_counts()}")
print(f"\nCurrency distribution:\n{df['currency'].value_counts()}")


## Cell 3 — Train / Val / Test split  80/10/10

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 3 — Train / Val / Test  80/10/10              ║
# ╚══════════════════════════════════════════════════════╝
X = df["raw_text"].values
y = df["label_id"].values

X_tr, X_tmp, y_tr, y_tmp, idx_tr, idx_tmp = train_test_split(
    X, y, df.index, test_size=0.20, random_state=42, stratify=y)
X_val, X_te, y_val, y_te, idx_val, idx_te = train_test_split(
    X_tmp, y_tmp, idx_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

df_tr  = df.loc[idx_tr].reset_index(drop=True)
df_val = df.loc[idx_val].reset_index(drop=True)
df_te  = df.loc[idx_te].reset_index(drop=True)

print(f"Train : {len(X_tr):>8,}")
print(f"Val   : {len(X_val):>8,}")
print(f"Test  : {len(X_te):>8,}")


## Cell 4 — EDA charts

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 4 — EDA charts                                ║
# ╚══════════════════════════════════════════════════════╝
COLORS = {"buy": "#10b981", "hold": "#f59e0b", "sell": "#f43f5e"}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

vc = df["impact"].value_counts()
axes[0].bar(vc.index, vc.values,
            color=[COLORS[l] for l in vc.index], edgecolor="none")
axes[0].set_title("Signal Distribution", fontweight="bold")
for i, v in enumerate(vc.values):
    axes[0].text(i, v + max(vc) * 0.01, f"{v:,}", ha="center", fontsize=9, fontweight="bold")

pivot = df.groupby(["currency", "impact"]).size().unstack(fill_value=0)
for col in ["buy", "hold", "sell"]:
    if col not in pivot.columns:
        pivot[col] = 0
pivot_pct = pivot[["buy", "hold", "sell"]].div(pivot[["buy", "hold", "sell"]].sum(axis=1), axis=0) * 100
sns.heatmap(pivot_pct, ax=axes[1], annot=True, fmt=".1f",
            cmap="RdYlGn", cbar_kws={"label": "%"}, linewidths=0.5)
axes[1].set_title("Signal % per Currency", fontweight="bold")

plt.tight_layout()
plt.savefig(MODELS_DIR / "eda.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → models/eda.png")


## Model 1 — Logistic Regression (TF-IDF)
**Why:** Fastest model, no GPU needed. Reliably catches high-signal phrases. Acts as a regulariser in the ensemble.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  MODEL 1 — Logistic Regression (TF-IDF)             ║
# ╚══════════════════════════════════════════════════════╝
print("=" * 55)
print("MODEL 1 — Logistic Regression (TF-IDF)")
print("=" * 55)

lr_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50_000, ngram_range=(1, 2), sublinear_tf=True,
        min_df=2, strip_accents="unicode", decode_error="replace")),
    ("clf", LogisticRegression(
        C=1.0, max_iter=1000, class_weight="balanced",
        solver="lbfgs", multi_class="multinomial", random_state=42)),
])
lr_pipe.fit(X_tr, y_tr)

lr_probs = lr_pipe.predict_proba(X_te)
lr_preds = lr_probs.argmax(axis=1)
lr_f1    = f1_score(y_te, lr_preds, average="weighted", zero_division=0)
lr_acc   = accuracy_score(y_te, lr_preds)

print(f"Test F1  : {lr_f1:.4f}")
print(f"Test Acc : {lr_acc:.4f}")
print(classification_report(y_te, lr_preds,
      target_names=["buy", "hold", "sell"], zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_te, lr_preds),
    display_labels=["buy", "hold", "sell"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("LR — Confusion Matrix", fontweight="bold")
plt.tight_layout(); plt.show()

# Save — protocol=2 ensures numpy 1.x/2.x cross-compatibility
safe_dump(lr_pipe,                       MODELS_DIR / "model_lr.pkl")
safe_dump(lr_pipe.named_steps["tfidf"],  MODELS_DIR / "tfidf_vectorizer.pkl")


## Model 2 — LightGBM
**Why:** Only model that uses the currency metadata + FinBERT score. Same phrase means different things for different currencies.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  MODEL 2 — LightGBM                                 ║
# ╚══════════════════════════════════════════════════════╝
print("=" * 55)
print("MODEL 2 — LightGBM")
print("=" * 55)

tfidf_vec = lr_pipe.named_steps["tfidf"]   # reuse vectorizer from Model 1

def build_extra(df_slice: pd.DataFrame) -> np.ndarray:
    """Extra features: finbert_score + one-hot currency."""
    score = df_slice["finbert_score"].values.reshape(-1, 1)
    ohe   = np.column_stack([
        (df_slice["currency"] == c).astype(np.float32).values
        for c in CURRENCIES
    ])
    return np.hstack([score, ohe])

lgb_X_tr  = hstack([tfidf_vec.transform(X_tr),  csr_matrix(build_extra(df_tr))])
lgb_X_val = hstack([tfidf_vec.transform(X_val), csr_matrix(build_extra(df_val))])
lgb_X_te  = hstack([tfidf_vec.transform(X_te),  csr_matrix(build_extra(df_te))])

lgb_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1)
lgb_model.fit(
    lgb_X_tr, y_tr,
    eval_set=[(lgb_X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)])

lgb_probs = lgb_model.predict_proba(lgb_X_te)
lgb_preds = lgb_probs.argmax(axis=1)
lgb_f1    = f1_score(y_te, lgb_preds, average="weighted", zero_division=0)
lgb_acc   = accuracy_score(y_te, lgb_preds)

print(f"Test F1  : {lgb_f1:.4f}")
print(f"Test Acc : {lgb_acc:.4f}")
print(classification_report(y_te, lgb_preds,
      target_names=["buy", "hold", "sell"], zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_te, lgb_preds),
    display_labels=["buy", "hold", "sell"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("LightGBM — Confusion Matrix", fontweight="bold")
plt.tight_layout(); plt.show()

safe_dump(lgb_model, MODELS_DIR / "model_lgb.pkl")


## Model 3 — TextCNN
**Why:** Fastest deep model. Detects n-gram patterns anywhere in text. Great for short headlines.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  MODEL 3a — Vocabulary & tokeniser                  ║
# ╚══════════════════════════════════════════════════════╝
VOCAB_SIZE   = 30_000
MAX_SEQ_LEN  = 128
EMBED_DIM    = 128
NUM_FILTERS  = 128
KERNEL_SIZES = [2, 3, 4]
BATCH_SIZE   = 64
EPOCHS_CNN   = 15

vocab_builder = CountVectorizer(
    max_features=VOCAB_SIZE - 2,
    token_pattern=r"\b[a-zA-Z][a-zA-Z0-9]*\b")
vocab_builder.fit(X_tr)
word2idx: dict[str, int] = {w: i + 2 for i, w in enumerate(vocab_builder.vocabulary_)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1
print(f"Vocabulary size: {len(word2idx):,}")


def tok_pad(texts, max_len: int = MAX_SEQ_LEN) -> torch.Tensor:
    out = []
    for t in texts:
        ids = [word2idx.get(w.lower(), 1) for w in str(t).split()]
        ids = ids[:max_len] + [0] * max(0, max_len - len(ids))
        out.append(ids)
    return torch.tensor(out, dtype=torch.long)


class TextDS(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


pin = DEVICE == "cuda"
X_tr_tok  = tok_pad(X_tr)
X_val_tok = tok_pad(X_val)
X_te_tok  = tok_pad(X_te)
train_dl  = DataLoader(TextDS(X_tr_tok,  y_tr),  batch_size=BATCH_SIZE,     shuffle=True, pin_memory=pin)
val_dl    = DataLoader(TextDS(X_val_tok, y_val), batch_size=BATCH_SIZE * 2, pin_memory=pin)
test_dl   = DataLoader(TextDS(X_te_tok,  y_te),  batch_size=BATCH_SIZE * 2, pin_memory=pin)
print("Dataloaders ready.")


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  MODEL 3b — TextCNN architecture + training         ║
# ╚══════════════════════════════════════════════════════╝
print("=" * 55)
print("MODEL 3 — TextCNN")
print("=" * 55)

class TextCNN(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                 num_filters=NUM_FILTERS, kernel_sizes=None,
                 num_classes=3, dropout=0.4):
        super().__init__()
        if kernel_sizes is None:
            kernel_sizes = KERNEL_SIZES
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])
        self.drop  = nn.Dropout(dropout)
        self.fc    = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        e = self.embed(x).permute(0, 2, 1)
        pooled = [torch.relu(c(e)).max(dim=2).values for c in self.convs]
        return self.fc(self.drop(torch.cat(pooled, dim=1)))

criterion = nn.CrossEntropyLoss()
cnn_model = TextCNN().to(DEVICE)
cnn_opt   = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)
cnn_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    cnn_opt, mode="max", patience=2, factor=0.5)

best_f1, best_ep = 0.0, 0
for ep in range(1, EPOCHS_CNN + 1):
    cnn_model.train(); tl = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        cnn_opt.zero_grad()
        loss = criterion(cnn_model(xb), yb)
        loss.backward(); cnn_opt.step(); tl += loss.item()

    cnn_model.eval(); vp = []
    with torch.no_grad():
        for xb, _ in val_dl:
            vp.extend(cnn_model(xb.to(DEVICE)).argmax(1).cpu().tolist())
    vf1 = f1_score(y_val, vp, average="weighted", zero_division=0)
    cnn_sched.step(vf1)
    if vf1 > best_f1:
        best_f1, best_ep = vf1, ep
        torch.save(cnn_model.state_dict(), MODELS_DIR / "model_textcnn.pt")
    if ep % 3 == 0 or ep == EPOCHS_CNN:
        print(f"Ep {ep:02d}/{EPOCHS_CNN}  loss={tl/len(train_dl):.4f}"
              f"  val_F1={vf1:.4f}" + (" ← best" if ep == best_ep else ""))

print(f"\nBest val F1: {best_f1:.4f}  (epoch {best_ep})")


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  MODEL 3c — Evaluate TextCNN                        ║
# ╚══════════════════════════════════════════════════════╝
cnn_model.load_state_dict(torch.load(
    MODELS_DIR / "model_textcnn.pt",
    map_location=DEVICE, weights_only=True))   # weights_only=True required in PyTorch ≥ 2.0
cnn_model.eval()

cnn_probs_parts, cnn_preds_list = [], []
with torch.no_grad():
    for xb, _ in test_dl:
        logits = cnn_model(xb.to(DEVICE))
        cnn_probs_parts.append(torch.softmax(logits, 1).cpu().numpy())
        cnn_preds_list.extend(logits.argmax(1).cpu().tolist())

cnn_probs = np.vstack(cnn_probs_parts)
cnn_preds = np.array(cnn_preds_list)
cnn_f1    = f1_score(y_te, cnn_preds, average="weighted", zero_division=0)
cnn_acc   = accuracy_score(y_te, cnn_preds)

print(f"Test F1  : {cnn_f1:.4f}")
print(f"Test Acc : {cnn_acc:.4f}")
print(classification_report(y_te, cnn_preds,
      target_names=["buy", "hold", "sell"], zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_te, cnn_preds),
    display_labels=["buy", "hold", "sell"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("TextCNN — Confusion Matrix", fontweight="bold")
plt.tight_layout(); plt.show()

# Save vocab — protocol=2
safe_dump({"word2idx": word2idx, "max_len": MAX_SEQ_LEN}, MODELS_DIR / "textcnn_vocab.pkl")
print("Saved → models/model_textcnn.pt  (weights saved during training)")


## Model 4 — BiLSTM + Attention
**Why:** Reads the full sentence in both directions. Catches negation and contrast that CNN misses.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  MODEL 4a — BiLSTM + Attention architecture         ║
# ╚══════════════════════════════════════════════════════╝
EPOCHS_LSTM = 15

class SelfAttention(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1, bias=False)
    def forward(self, lstm_out):
        weights = torch.softmax(self.attn(lstm_out), dim=1)
        return (lstm_out * weights).sum(dim=1)

class BiLSTMAttn(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                 hidden_dim=128, n_layers=2, num_classes=3, dropout=0.4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm  = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                             bidirectional=True, batch_first=True,
                             dropout=dropout if n_layers > 1 else 0.0)
        self.attn  = SelfAttention(hidden_dim)
        self.drop  = nn.Dropout(dropout)
        self.fc    = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.embed(x)
        out, _ = self.lstm(emb)
        return self.fc(self.drop(self.attn(out)))

print("BiLSTM + Attention defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  MODEL 4b — Train BiLSTM                            ║
# ╚══════════════════════════════════════════════════════╝
print("=" * 55)
print("MODEL 4 — BiLSTM + Attention")
print("=" * 55)

lstm_model = BiLSTMAttn().to(DEVICE)
lstm_opt   = torch.optim.AdamW(lstm_model.parameters(), lr=5e-4, weight_decay=1e-4)
lstm_sched = torch.optim.lr_scheduler.CosineAnnealingLR(lstm_opt, T_max=EPOCHS_LSTM)
best_f1, best_ep = 0.0, 0

for ep in range(1, EPOCHS_LSTM + 1):
    lstm_model.train(); tl = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        lstm_opt.zero_grad()
        loss = criterion(lstm_model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        lstm_opt.step(); tl += loss.item()
    lstm_sched.step()

    lstm_model.eval(); vp = []
    with torch.no_grad():
        for xb, _ in val_dl:
            vp.extend(lstm_model(xb.to(DEVICE)).argmax(1).cpu().tolist())
    vf1 = f1_score(y_val, vp, average="weighted", zero_division=0)
    if vf1 > best_f1:
        best_f1, best_ep = vf1, ep
        torch.save(lstm_model.state_dict(), MODELS_DIR / "model_bilstm.pt")
    if ep % 3 == 0 or ep == EPOCHS_LSTM:
        print(f"Ep {ep:02d}/{EPOCHS_LSTM}  loss={tl/len(train_dl):.4f}"
              f"  val_F1={vf1:.4f}" + (" ← best" if ep == best_ep else ""))

print(f"\nBest val F1: {best_f1:.4f}  (epoch {best_ep})")


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  MODEL 4c — Evaluate BiLSTM                         ║
# ╚══════════════════════════════════════════════════════╝
lstm_model.load_state_dict(torch.load(
    MODELS_DIR / "model_bilstm.pt",
    map_location=DEVICE, weights_only=True))
lstm_model.eval()

lstm_probs_parts, lstm_preds_list = [], []
with torch.no_grad():
    for xb, _ in test_dl:
        logits = lstm_model(xb.to(DEVICE))
        lstm_probs_parts.append(torch.softmax(logits, 1).cpu().numpy())
        lstm_preds_list.extend(logits.argmax(1).cpu().tolist())

lstm_probs = np.vstack(lstm_probs_parts)
lstm_preds = np.array(lstm_preds_list)
lstm_f1    = f1_score(y_te, lstm_preds, average="weighted", zero_division=0)
lstm_acc   = accuracy_score(y_te, lstm_preds)

print(f"Test F1  : {lstm_f1:.4f}")
print(f"Test Acc : {lstm_acc:.4f}")
print(classification_report(y_te, lstm_preds,
      target_names=["buy", "hold", "sell"], zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_te, lstm_preds),
    display_labels=["buy", "hold", "sell"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("BiLSTM+Attention — Confusion Matrix", fontweight="bold")
plt.tight_layout(); plt.show()
print("Saved → models/model_bilstm.pt")


## Model 5 — DistilBERT fine-tuned
**Why:** 40% of FinBERT's parameters, same accuracy. Pre-trained on financial language. Earns 40% of the ensemble vote.

> **Note on warnings:** `UNEXPECTED keys` (vocab_transform etc.) = normal — those are the MLM head we don't need. `MISSING keys` (classifier) = normal — those are our new classification layers. Both warnings are expected every time.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  MODEL 5a — Tokenise for DistilBERT                 ║
# ╚══════════════════════════════════════════════════════╝
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np

DISTIL_NAME  = "distilbert-base-uncased"
DISTIL_DIR   = str(MODELS_DIR / "distilbert_finetuned")
BERT_MAXLEN  = 128
BERT_BATCH   = 32      # fine for Colab free GPU
BERT_EPOCHS  = 3

print(f"Loading tokenizer: {DISTIL_NAME} ...")
bert_tokenizer = AutoTokenizer.from_pretrained(DISTIL_NAME)


class BertDS(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len: int):
        enc = tokenizer(
            [str(t) for t in texts],
            truncation=True, padding="max_length",
            max_length=max_len, return_tensors="pt")
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels         = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {
            "input_ids":      self.input_ids[i],
            "attention_mask": self.attention_mask[i],
            "labels":         self.labels[i],
        }


bert_tr_ds  = BertDS(X_tr,  y_tr,  bert_tokenizer, BERT_MAXLEN)
bert_val_ds = BertDS(X_val, y_val, bert_tokenizer, BERT_MAXLEN)
bert_te_ds  = BertDS(X_te,  y_te,  bert_tokenizer, BERT_MAXLEN)
print(f"Train {len(bert_tr_ds):,}  Val {len(bert_val_ds):,}  Test {len(bert_te_ds):,}")


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  MODEL 5b — Load DistilBERT & fine-tune             ║
# ╚══════════════════════════════════════════════════════╝
print("=" * 55)
print("MODEL 5 — DistilBERT fine-tuned")
print("=" * 55)
print("Warnings about UNEXPECTED/MISSING keys are NORMAL — ignore them.\n")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1":  float(f1_score(labels, preds, average="weighted", zero_division=0)),
        "acc": float(accuracy_score(labels, preds)),
    }


# ignore_mismatched_sizes=True suppresses the size warning from the new head
bert_model = AutoModelForSequenceClassification.from_pretrained(
    DISTIL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

training_args = TrainingArguments(
    output_dir=DISTIL_DIR,
    num_train_epochs=BERT_EPOCHS,
    per_device_train_batch_size=BERT_BATCH,
    per_device_eval_batch_size=BERT_BATCH * 2,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",      # correct name for transformers >= 4.30
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=(DEVICE == "cuda"),
    dataloader_num_workers=0,
    logging_steps=50,
    report_to="none",           # disable wandb / mlflow
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=bert_tr_ds,
    eval_dataset=bert_val_ds,
    compute_metrics=compute_metrics,
)

print("Training ...  (~6 min on Colab T4 GPU, ~2h on CPU)")
trainer.train()
print("\n✓ Training complete.")


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  MODEL 5c — Evaluate & save DistilBERT              ║
# ╚══════════════════════════════════════════════════════╝
print("Evaluating on test set ...")
bert_out  = trainer.predict(bert_te_ds)
logits    = bert_out.predictions

# manual softmax (avoids scipy dependency)
ex         = np.exp(logits - logits.max(axis=1, keepdims=True))
bert_probs = ex / ex.sum(axis=1, keepdims=True)
bert_preds = logits.argmax(axis=1)

bert_f1  = f1_score(y_te, bert_preds, average="weighted", zero_division=0)
bert_acc = accuracy_score(y_te, bert_preds)

print(f"Test F1  : {bert_f1:.4f}")
print(f"Test Acc : {bert_acc:.4f}")
print(classification_report(y_te, bert_preds,
      target_names=["buy", "hold", "sell"], zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_te, bert_preds),
    display_labels=["buy", "hold", "sell"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("DistilBERT Fine-tuned — Confusion Matrix", fontweight="bold")
plt.tight_layout(); plt.show()

# Save model + tokenizer in HuggingFace format (not pickle — no numpy issue)
bert_model.save_pretrained(DISTIL_DIR)
bert_tokenizer.save_pretrained(DISTIL_DIR)
print(f"\n✓ Saved → {DISTIL_DIR}/")


## Ensemble — Weighted vote across all 5 models

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  ENSEMBLE — Weighted vote                           ║
# ╚══════════════════════════════════════════════════════╝
print("=" * 55)
print("ENSEMBLE")
print("=" * 55)

# Weights reflect model quality + speed tradeoff
WEIGHTS = {
    "distilbert": 0.40,   # strongest, financial pre-training
    "bilstm":     0.25,   # reads full context + attention
    "textcnn":    0.15,   # fast, great on headlines
    "lgb":        0.12,   # currency-aware
    "lr":         0.08,   # baseline regulariser
}

N = len(y_te)
for name, arr in [("lr", lr_probs), ("lgb", lgb_probs), ("cnn", cnn_probs),
                   ("lstm", lstm_probs), ("bert", bert_probs)]:
    assert arr.shape == (N, 3), f"{name} shape mismatch: {arr.shape} != ({N}, 3)"

ensemble_probs = (
    WEIGHTS["distilbert"] * bert_probs  +
    WEIGHTS["bilstm"]     * lstm_probs  +
    WEIGHTS["textcnn"]    * cnn_probs   +
    WEIGHTS["lgb"]        * lgb_probs   +
    WEIGHTS["lr"]         * lr_probs
)
ensemble_preds = ensemble_probs.argmax(axis=1)

ens_f1  = f1_score(y_te, ensemble_preds, average="weighted", zero_division=0)
ens_acc = accuracy_score(y_te, ensemble_preds)
print(f"Test F1  : {ens_f1:.4f}")
print(f"Test Acc : {ens_acc:.4f}")
print(classification_report(y_te, ensemble_preds,
      target_names=["buy", "hold", "sell"], zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_te, ensemble_preds),
    display_labels=["buy", "hold", "sell"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Ensemble — Confusion Matrix", fontweight="bold")
plt.tight_layout(); plt.show()

safe_dump(WEIGHTS, MODELS_DIR / "ensemble_weights.pkl")


## Comparison chart

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  Model comparison chart                             ║
# ╚══════════════════════════════════════════════════════╝
scores = {
    "LR\n(TF-IDF)":        lr_f1,
    "LightGBM":             lgb_f1,
    "TextCNN":              cnn_f1,
    "BiLSTM\n+Attn":       lstm_f1,
    "DistilBERT\nfine-tuned": bert_f1,
    "Ensemble":             ens_f1,
}
cols = ["#6b7280", "#6b7280", "#3b82f6", "#3b82f6", "#8b5cf6", "#10b981"]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(scores.keys(), scores.values(), color=cols, edgecolor="none", width=0.6)
for bar, val in zip(bars, scores.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.004,
            f"{val:.3f}", ha="center", fontweight="bold", fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Weighted F1")
ax.set_title("Model Comparison — FX Sentiment (v3)", fontweight="bold", fontsize=13)
ax.axhline(ens_f1, color="#10b981", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(MODELS_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → models/model_comparison.png")


## Final summary

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  FINAL SUMMARY                                      ║
# ╚══════════════════════════════════════════════════════╝
print()
print("=" * 55)
print("  ALL MODELS TRAINED (v3)")
print("=" * 55)

total_mb = 0
for f in sorted(MODELS_DIR.rglob("*")):
    if f.is_file():
        mb = f.stat().st_size / 1e6
        total_mb += mb
        print(f"  {str(f.relative_to(MODELS_DIR)):<44s}  {mb:6.1f} MB")

print(f"\n  Total: {total_mb:.1f} MB")
print()

for name, f1 in [
    ("LR (TF-IDF)",          lr_f1),
    ("LightGBM",             lgb_f1),
    ("TextCNN",              cnn_f1),
    ("BiLSTM + Attention",   lstm_f1),
    ("DistilBERT fine-tuned",bert_f1),
    ("ENSEMBLE",             ens_f1),
]:
    marker = "  ← final signal" if name == "ENSEMBLE" else ""
    print(f"  {name:<28s}  F1 = {f1:.4f}{marker}")

print()
print("All .pkl files saved with protocol=2")
print("→ They will load correctly on any Python version, any numpy version.")
print("→ No patching, no resave_models.py needed.")


## Download all models (run this last)
This zips everything and triggers a browser download.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  DOWNLOAD — zip + browser download                  ║
# ║  Run this cell AFTER the summary cell above.        ║
# ╚══════════════════════════════════════════════════════╝
import shutil
from pathlib import Path
import google.colab.files as colab_files

MODELS_DIR = Path("models")
ZIP_NAME   = "models_v3"

# ── 1. Verify all files are present ──────────────────────────────────────────
REQUIRED = [
    "model_lr.pkl",
    "tfidf_vectorizer.pkl",
    "model_lgb.pkl",
    "model_textcnn.pt",
    "textcnn_vocab.pkl",
    "model_bilstm.pt",
    "distilbert_finetuned/config.json",
    "distilbert_finetuned/model.safetensors",
    "ensemble_weights.pkl",
]

print("Checking files before zipping ...\n")
missing = [f for f in REQUIRED if not (MODELS_DIR / f).exists()]
if missing:
    print("ERROR — these files are missing:")
    for f in missing:
        print(f"  ✗  models/{f}")
    print("\nFix: make sure all training cells ran without errors, then re-run this cell.")
    raise FileNotFoundError("Missing model files — see above.")

print(f"✓ All {len(REQUIRED)} required files found.\n")

# ── 2. Show file sizes ────────────────────────────────────────────────────────
total_mb = 0
for f in sorted(MODELS_DIR.rglob("*")):
    if f.is_file():
        mb = f.stat().st_size / 1e6
        total_mb += mb
        print(f"  {str(f.relative_to(MODELS_DIR)):<46s}  {mb:6.1f} MB")

print(f"\n  Total uncompressed: {total_mb:.1f} MB")

# ── 3. Zip ────────────────────────────────────────────────────────────────────
print(f"\nZipping to {ZIP_NAME}.zip ...")
zip_path = shutil.make_archive(ZIP_NAME, "zip", root_dir=".", base_dir="models")
zip_mb   = Path(zip_path).stat().st_size / 1e6
print(f"✓ Zip created: {zip_path}  ({zip_mb:.1f} MB compressed)")

# ── 4. Download ───────────────────────────────────────────────────────────────
print("\nStarting download ... (check your browser downloads bar)")
colab_files.download(zip_path)
print("\n✓ Done!")
print()
print("After downloading, on your PC:")
print("  1.  Unzip models_v3.zip  →  you get a models/ folder")
print("  2.  Put models/ next to predict.py")
print("  3.  Run:  make check     (verifies all files)")
print("  4.  Run:  make run       (first live prediction)")
